<a href="https://colab.research.google.com/github/ochimasato0186/FE2024L0301codespace/blob/main/SD4AI%E6%BC%94%E7%BF%9204%EF%BC%BF%E8%B6%8A%E6%99%BA%E3%80%80%E7%A5%90%E4%BB%81.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -U -q google-generativeai google-ai-generativelanguage
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY= userdata.get("GOOGLE_API_KEY")
genai.configure(api_key=GOOGLE_API_KEY)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
from IPython.core.display import json
import json

def get_current_weather(location,unit="fahrenheit"):
   """　指定された場所の現在の天気を取得する関数 """
   if "tokyo" in location.lower():
     rt = json.dumps({
         "location":"Tokyo",
         "temperature":"10",
         "unit":"celsius"
     })
     return rt
   elif "san francisco" in location.lower():
     rt = json.dumps({
         "location":"San Francisco",
         "temperature":"72",
         "unit":"fahrenheit"
     })
     return rt
   elif "paris" in location.lower():
     rt = json.dumps({
         "location":"Paris",
         "temperature":"22",
         "unit":"celsius"
     })
     return rt
   else:
     rt = json.dumps({
         "location":location,
         "error":"unknown location"
     },ensure_ascii=False)

# **Gemini実装：API呼び出し**

In [4]:
model = genai.GenerativeModel(
    model_name="gemini-2.5-flash",
    tools=[get_current_weather],
    tool_config={'function_calling_config':{"mode":"AUTO"}}
)

chat = model.start_chat()

response = chat.send_message("tokyoの気温はどうですか？")
tools_config={"function_calling_config":{"mode":"ANY"}}
print("tools_call の結果：実行する関数名と因数が指定されている")
print(response.candidates[0].content.parts[0].function_call)

tools_call の結果：実行する関数名と因数が指定されている
name: "get_current_weather"
args {
  fields {
    key: "location"
    value {
      string_value: "Tokyo"
    }
  }
}



# **応答で帰ってきた情報を全て表示**

In [5]:
print(response)

response:
GenerateContentResponse(
    done=True,
    iterator=None,
    result=protos.GenerateContentResponse({
      "candidates": [
        {
          "content": {
            "parts": [
              {
                "function_call": {
                  "name": "get_current_weather",
                  "args": {
                    "location": "Tokyo"
                  }
                }
              }
            ],
            "role": "model"
          },
          "finish_reason": "STOP",
          "index": 0
        }
      ],
      "usage_metadata": {
        "prompt_token_count": 70,
        "candidates_token_count": 17,
        "total_token_count": 167
      },
      "model_version": "gemini-2.5-flash"
    }),
)


# **レスポンスデータのうちの、function callの情報を変数に代入して、ローカルの関数を実行する**

In [7]:
function_call = response.candidates[0].content.parts[0].function_call

if function_call:
  function_args = function_call.args

  function_response_str = get_current_weather(
      location=function_args['location'],
      unit=function_args.get('unit')
  )
  print(f"ローカル関数呼び出しの結果：\n{function_response_str}")

ローカル関数呼び出しの結果：
{"location": "Tokyo", "temperature": "10", "unit": "celsius"}


# **ローカル関数の結果をLLMに伝えて、AIに最終的な返事を返してもらう**

In [9]:
second_response=chat.send_message({
      "function_response":{
          "id":function_call.id,
          "name":function_call.name,
          "response":{"result":function_response_str}
      }
  })
print(f"2度目の呼び出しの結果応答：{second_response}")
print(f"応答に含まれたメッセージ：{second_response.text}")

2度目の呼び出しの結果応答：response:
GenerateContentResponse(
    done=True,
    iterator=None,
    result=protos.GenerateContentResponse({
      "candidates": [
        {
          "content": {
            "parts": [
              {
                "text": "\u6771\u4eac\u306e\u73fe\u5728\u306e\u6c17\u6e29\u306f10\u5ea6\u3067\u3059\u3002"
              }
            ],
            "role": "model"
          },
          "finish_reason": "STOP",
          "index": 0
        }
      ],
      "usage_metadata": {
        "prompt_token_count": 124,
        "candidates_token_count": 10,
        "total_token_count": 134
      },
      "model_version": "gemini-2.5-flash"
    }),
)
応答に含まれたメッセージ：東京の現在の気温は10度です。
